# Chẩn đoán ung thư vú bằng Machine Learning

## Bài toán
Phân loại nhị phân: dự đoán khối u là lành tính (Benign) hay ác tính (Malignant) dựa trên 30 đặc trưng số đo từ ảnh sinh thiết kim nhỏ.

**Bản cải tiến so với bản gốc:**
- Gộp `StandardScaler` + model vào chung một `Pipeline` (tránh quên áp dụng lại scaler lúc deploy).
- Thêm `GridSearchCV` tinh chỉnh siêu tham số cho cả 5 model thay vì dùng tham số cố định.
- Tính thêm Accuracy trên tập train để phát hiện overfit/underfit (so sánh train vs test).
- Lưu (`savefig`) đầy đủ tất cả hình EDA + confusion matrix vào `docs/figures` (bản gốc chỉ `plt.show()`, không lưu file).
- Dọn các cell bị lặp lại (drop cột + tách X/y bị làm 2 lần).
- Thêm bước chọn model tốt nhất kèm lý do, và đóng gói `model.joblib` + `schema.json` + `metadata.json` để AI Service dùng trực tiếp.


In [ ]:
import zipfile
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)

sns.set_theme(style="whitegrid")
print("Các thư viện đã được import thành công!")

In [ ]:
!git clone https://github.com/YoungHyyy/18_10123154_12523066_CDUTV.git

In [ ]:
ROOT = Path("/content/18_10123154_12523066_CDUTV")
DATA_ZIP = ROOT / "ai-models" / "data" / "dataset.zip"
FIGURES_PATH = ROOT / "docs" / "figures"
MODEL_DIR = ROOT / "ai-models" / "models"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DATA_ZIP, "r") as z:
    df = pd.read_csv(z.open("data.csv"))

print("Dataset shape:", df.shape)
df.head()

## 1. Khám phá & làm sạch dữ liệu

In [ ]:
print("===== THÔNG TIN DATASET =====")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

print("\n===== KIỂU DỮ LIỆU =====")
print(df.dtypes)

print("\n===== MISSING VALUES (trước khi làm sạch) =====")
print(df.isnull().sum())

print("\n===== CỘT id / Unnamed: 32 =====")
print("Số giá trị khác nhau của id:", df["id"].nunique())
print("Unnamed: 32 toàn bộ NaN:", df["Unnamed: 32"].isnull().all())

In [ ]:
# Làm sạch: bỏ cột không dùng cho model — chỉ làm 1 lần duy nhất (bản gốc bị lặp 2 lần)
df_clean = df.drop(columns=["id", "Unnamed: 32"])

X = df_clean.drop(columns=["diagnosis"])
y = df_clean["diagnosis"]  # giữ dạng nhãn chữ 'B'/'M'

print("Kích thước sau khi làm sạch:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Số dòng trùng lặp:", df_clean.duplicated().sum())
print("X shape:", X.shape, "| y shape:", y.shape)

feature_cols = list(X.columns)
print(f"\nSố đặc trưng: {len(feature_cols)}")

print("\nPhân bố nhãn:")
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

## 2. Phân tích khám phá dữ liệu (EDA) — lưu toàn bộ hình vào docs/figures

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_clean, x="diagnosis", order=["B", "M"])
ax.set_title("Phân bố chẩn đoán khối u")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Số lượng mẫu")
for c in ax.containers:
    ax.bar_label(c)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "01_target_distribution.png", dpi=150)
plt.show()

In [ ]:
features_to_plot = [
    "radius_mean", "texture_mean", "perimeter_mean",
    "area_mean", "smoothness_mean", "compactness_mean",
]

for feature in features_to_plot:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df_clean, x=feature, hue="diagnosis", kde=True)
    plt.title(f"Phân bố {feature} theo chẩn đoán")
    plt.xlabel(feature)
    plt.ylabel("Số lượng")
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / f"02_hist_{feature}.png", dpi=150)
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_clean, x="diagnosis", y="radius_mean")
plt.title("Phân bố radius_mean theo chẩn đoán")
plt.xlabel("Diagnosis")
plt.ylabel("radius_mean")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "03_boxplot_radius_mean.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))
corr = X.corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Ma trận tương quan giữa các đặc trưng")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "04_correlation_heatmap.png", dpi=150)
plt.show()

corr_abs = corr.abs()
pairs = (
    corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
    .stack().sort_values(ascending=False)
)
print("Top 5 cặp đặc trưng tương quan cao nhất:")
print(pairs.head(5))

In [ ]:
plt.figure(figsize=(10, 3))
miss = X.isnull().sum()
plt.bar(range(len(miss)), miss.values)
plt.xticks([])
plt.title(f"Số giá trị thiếu theo từng cột (tổng = {int(miss.sum())})")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "05_missing_values.png", dpi=150)
plt.show()

## 3. Chia train/test

- 80% train / 20% test, `stratify=y` để giữ nguyên tỉ lệ B/M ở cả 2 tập (dữ liệu mất cân bằng nhẹ).
- **Không** chuẩn hóa (scale) riêng ở đây — scaler sẽ được gộp vào Pipeline ở bước huấn luyện để tránh rò rỉ dữ liệu và để deploy được nguyên khối.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nPhân bố y_train:")
print(y_train.value_counts(normalize=True))
print("\nPhân bố y_test:")
print(y_test.value_counts(normalize=True))

## 4. Huấn luyện 5 model với GridSearchCV

Mỗi model là một `Pipeline` (StandardScaler + classifier với cây quyết định/rừng không cần scale nhưng để chung Pipeline cho đồng nhất khi deploy), tinh chỉnh bằng `GridSearchCV(cv=5, scoring='f1_macro')`.

In [ ]:
candidates = {
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, random_state=42))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "KNN": (
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7, 9, 11]},
    ),
    "SVM": (
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", probability=True, random_state=42))]),
        {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", "auto"]},
    ),
    "Decision Tree": (
        Pipeline([("clf", DecisionTreeClassifier(random_state=42))]),
        {"clf__max_depth": [3, 5, 7, None], "clf__min_samples_leaf": [1, 5, 10]},
    ),
    "Random Forest": (
        Pipeline([("clf", RandomForestClassifier(random_state=42))]),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 5, 10]},
    ),
}

results = []
fitted = {}

for name, (pipe, grid) in candidates.items():
    search = GridSearchCV(pipe, grid, cv=5, scoring="f1_macro", n_jobs=-1)
    t0 = time.time()
    search.fit(X_train, y_train)
    train_time = time.time() - t0
    best = search.best_estimator_
    fitted[name] = best

    y_pred = best.predict(X_test)
    malignant_idx = list(best.classes_).index("M")
    y_proba = best.predict_proba(X_test)[:, malignant_idx]

    acc_train = accuracy_score(y_train, best.predict(X_train))
    acc_test = accuracy_score(y_test, y_pred)
    y_test_binary = (y_test == "M").astype(int)

    metrics = {
        "Model": name,
        "best_params": search.best_params_,
        "Accuracy_train": round(acc_train, 4),
        "Accuracy": round(acc_test, 4),
        "Precision": round(precision_score(y_test, y_pred, pos_label="M"), 4),
        "Recall": round(recall_score(y_test, y_pred, pos_label="M"), 4),
        "F1-score": round(f1_score(y_test, y_pred, pos_label="M"), 4),
        "ROC-AUC": round(roc_auc_score(y_test_binary, y_proba), 4),
        "overfit_gap": round(acc_train - acc_test, 4),
        "train_time_s": round(train_time, 3),
    }
    results.append(metrics)
    print(f"✓ {name}: {metrics}")

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
results_df

## 5. Confusion matrix cho từng model (lưu file)

In [ ]:
for name, best in fitted.items():
    y_pred = best.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["B", "M"], yticklabels=["B", "M"])
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.tight_layout()
    safe_name = name.replace(" ", "_").lower()
    plt.savefig(FIGURES_PATH / f"cm_{safe_name}.png", dpi=150)
    plt.show()

## 6. ROC Curve & biểu đồ so sánh metric (lưu file)

In [ ]:
plt.figure(figsize=(10, 7))
for name, best in fitted.items():
    malignant_idx = list(best.classes_).index("M")
    y_proba = best.predict_proba(X_test)[:, malignant_idx]
    y_test_binary = (y_test == "M").astype(int)
    fpr, tpr, _ = roc_curve(y_test_binary, y_proba)
    auc = roc_auc_score(y_test_binary, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random classifier (AUC = 0.5000)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve - So sánh 5 mô hình")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "roc_curve_5_models.png", dpi=150)
plt.show()

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score"]
model_names = results_df["Model"].values
x = np.arange(len(model_names))
width = 0.2

plt.figure(figsize=(14, 7))
for i, metric in enumerate(metrics_to_plot):
    values = results_df[metric].values
    bars = plt.bar(x + (i - 1.5) * width, values, width, label=metric)
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{value:.3f}",
                  ha="center", va="bottom", fontsize=8)

plt.xlabel("Mô hình")
plt.ylabel("Giá trị")
plt.title("So sánh Accuracy, Precision, Recall và F1-score của 5 mô hình")
plt.xticks(x, model_names, rotation=15)
plt.ylim(0, 1.15)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "model_metrics_comparison.png", dpi=150)
plt.show()

## 7. Chọn model tốt nhất & đóng gói

Tiêu chí chọn: ưu tiên model có `overfit_gap` (Accuracy train − Accuracy test) nhỏ (< 0.05, tức không lệch nhiều giữa train/test), sau đó chọn F1-score cao nhất trong số đó — tránh chọn nhầm model chỉ "học thuộc" tập train (điển hình là Decision Tree không giới hạn độ sâu).

In [ ]:
candidates_ok = results_df[results_df["overfit_gap"] < 0.05].sort_values("F1-score", ascending=False)
best_row = candidates_ok.iloc[0] if len(candidates_ok) else results_df.iloc[0]
best_name = best_row["Model"]
best_pipe = fitted[best_name]

print(f">>> Model được chọn: {best_name}")
print(best_row)

# Lưu model (Pipeline gồm cả scaler — không cần xử lý riêng lúc deploy)
joblib.dump(best_pipe, MODEL_DIR / "model.joblib")

metadata = {
    "model_name": best_name,
    "best_params": best_row["best_params"],
    "metrics_test": {
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1": float(best_row["F1-score"]),
        "roc_auc": float(best_row["ROC-AUC"]),
    },
    "overfit_gap_train_minus_test_accuracy": float(best_row["overfit_gap"]),
    "dataset": "Breast Cancer Wisconsin (Diagnostic) - Kaggle uciml/breast-cancer-wisconsin-data",
    "n_features": len(feature_cols),
    "positive_class": "M (malignant)",
    "trained_at": pd.Timestamp.now().isoformat(),
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

schema = {"features": feature_cols, "target": "diagnosis (B=lành tính, M=ác tính)"}
(MODEL_DIR / "schema.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False))

results_df.to_csv(MODEL_DIR / "metrics_comparison.csv", index=False)

print("\n✓ Đã lưu model.joblib, schema.json, metadata.json vào", MODEL_DIR)
print("✓ Đã lưu toàn bộ hình EDA + confusion matrix + ROC + so sánh metric vào", FIGURES_PATH)
print("\nBước tiếp theo: commit + push các file này lên nhánh của bạn, rồi rebuild AI Service (docker compose build ai-service).")